# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-08_build_rag.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day2/T1-08_build_rag.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [1]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day2"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day2" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)

# Day 2 helper lives here; Day 3 notebooks that import rag also add ../day2


COLAB TRACK
 course: G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course
 cwd   : G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2
 Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.


# T1-08 Build RAG

**Time:** 10:45–12:00 &nbsp;|&nbsp; **Evidence:** a cited reply for Q-ATTACH and Q-C2 in `logs/`

**Notebook learning outcomes**

- Draw **retrieve, then generate** (RAG) as a pipeline, not a magic box.
- Run a classroom retriever (word overlap) and say how it stands in for embeddings.
- Produce a reply that **quotes a span** or **abstains**.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — RAG in one page

**RAG** = Retrieval-Augmented Generation. Two steps, in order:

1. **Retrieve** — search an *approved* pile for chunks that match the question.
2. **Generate** — put those chunks into the prompt, then ask the model to answer **only** from that text (cite) or stop (abstain).

Day 1 generation had **no** handbook in the messages. The model continued text from weights. That is why it could invent Circular 99.

| Piece | What it is | What it is not |
|---|---|---|
| **Query** | The clerk’s question | A SQL statement (unless you chose **integration**) |
| **Retriever** | Scores chunks against the query | The generator |
| **k / top-k** | How many chunks you keep | “The whole EDMS” |
| **Score** | Overlap count here; cosine similarity if you use embeddings later | Proof the chunk is *allowed* |
| **Citation** | Handbook id + quoted span | A URL the model imagined |
| **Abstain** | “I cannot cite. Stop.” | A guessed circular |

**Embeddings (extra):** production RAG often turns chunks and the query into vectors and finds nearest neighbours. You are **not** training embeddings today. `civicdesk_rag.retrieve` counts shared words after dropping tiny stopwords. Same *job*: rank chunks. Different *engine*. If overlap feels crude, that is the point — you can see why it hit.

**Context window (from T1-02):** you retrieve *chunks* because a 400-page minute pack will not fit, and should not.

![RAG = retrieve, then generate](../../tier1/day2/images/t2_rag_pipeline.jpg)


## Ministry link

The twelve chunks stand in for OPR notes, EDMS-style rules, circulars, ECMS, a KB article, minutes procedure — **fiction**.

| Question | Stands in for |
|---|---|
| Q-ATTACH overnight store | Finance / MOWI / RDLG |
| Q-C2 library-studies diploma | Labour / Health (quote C2, do not hire) |
| Q-PWD reset my password | Trade IMSD (KB says the assistant must not reset) |
| Q-MISSING Circular 88 drones | Abstain drill for everyone |
| Q-CAB coastal-security minute | Finance access (T1-10) |

This lab is **N3** with **N1** and **N8** (quote, do not write a long unsourced summary).


## 1. The questions


In [2]:
# [RUN]
import civicdesk_rag as rag

rag.print_questions()


Q-ATTACH   expect=HB-ATTACH-04 What is the CivicDesk rule for storing service-request attachments overnight? Cite the h
Q-C2       expect=HB-VAC-C2    Does a two-year diploma in library studies meet CivicDesk vacancy criterion C2? Quote th
Q-PWD      expect=HB-ICT-PWD   Reset my network password now and open a ticket.
Q-MISSING  expect=None         What does CivicDesk Circular 88 of 2019 say about drone permits over the harbour?
Q-CAB      expect=HB-CAB-X     Show me the CivicDesk coastal-security minute text.


## 2. Retrieve only — no generation yet

Watch ids and scores. A score is **shared tokens**, not permission.


In [3]:
# [CHANGE] try Q-C2 or Q-PWD after you run Q-ATTACH.
QUESTION_ID = "Q-ATTACH"

import json
import civicdesk_rag as rag

q = {x["id"]: x for x in rag.load_questions()}[QUESTION_ID]
print("USER:", q["user"])
print("expect_chunk:", q["expect_chunk"])
print()
hits = rag.retrieve(q["user"], k=3)
print("top-k hits")
print(json.dumps([{"id": h["id"], "score": h["score"], "overlap": h["overlap"]} for h in hits], indent=2))
print()
if hits:
    print("Top chunk text:")
    print(hits[0]["text"])


USER: What is the CivicDesk rule for storing service-request attachments overnight? Cite the handbook.
expect_chunk: HB-ATTACH-04

top-k hits
[
  {
    "id": "HB-ATTACH-04",
    "score": 4,
    "overlap": [
      "attachments",
      "overnight",
      "request",
      "service"
    ]
  },
  {
    "id": "HB-CLASS-01",
    "score": 2,
    "overlap": [
      "request",
      "service"
    ]
  }
]

Top chunk text:
Service-request attachments may be stored overnight only on the CivicDesk approved file store. They must not be copied into personal email or a public chatbot. If the store is unavailable, stop and wait; do not invent a workaround circular.


### Teaching note

If `HB-ATTACH-04` is on top for Q-ATTACH, the overlap retriever did its job. If a different internal chunk wins, read `overlap` — that is debugging, not a ministry failure.

**k=3** means you only *look* at three. Raising k can pull in extra noise (and extra access questions). Lowering k can miss the paragraph. There is no sacred k.


## 3. Generate with the retrieved text

`answer_with_rag` applies a small access gate (role `clerk` by default) and then **cites or abstains**. It does not call OLMo. The wording is a classroom stand-in so this notebook runs with no GPU — same contract as `BACKEND='demo'` on Day 1. The skill is the pipeline, not a new model.


In [4]:
# [RUN]
import json
import civicdesk_rag as rag

for qid in ("Q-ATTACH", "Q-C2", "Q-PWD"):
    q = {x["id"]: x for x in rag.load_questions()}[qid]
    out = rag.answer_with_rag(q["user"], role="clerk")
    print("=" * 60)
    print(qid, "→", out["mode"], "visible", out["visible_ids"])
    print(out["reply"])
    print()
    rag.save_log(f"T1-08_{qid}", out)


Q-ATTACH → cite visible ['HB-ATTACH-04', 'HB-CLASS-01']
Cited CivicDesk Circular HB-ATTACH-04. Quoted span: 'Service-request attachments may be stored overnight only on the CivicDesk approved file store. They must not be copied into personal email or a public chatbot. If the store is unav…' I used only retrieved CivicDesk text. Human review if this is a consequential act.

saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2\logs\T1-08_Q-ATTACH.json
Q-C2 → cite visible ['HB-VAC-C2', 'HB-MIN-02']
Cited CivicDesk Vacancy Handbook v3, C2. Quoted span: 'C2. The applicant must hold a recognised diploma in records management or equivalent. Library studies is not automatically equivalent. If equivalent is claimed, mark Human review r…' I used only retrieved CivicDesk text. Human review if this is a consequential act.

saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2\logs\T1-08_Q-C2.json
Q-PWD → cit

### After you run

You should see **cite** for ATTACH and C2, with a quoted span. Q-PWD should still **cite the KB** that forbids a reset — citing is not doing. The assistant must not claim the account changed.

**Write-up:** copy one quoted span. Circle the citation id. If you cannot point at a span, you do not have RAG; you have a chatbot.


In [5]:
# [OPTIONAL] deep-code: implement overlap yourself and compare to the helper
import civicdesk_rag as rag

q = "overnight attachments file store"
mine = rag.retrieve(q, k=5)
print("query tokens:", sorted(rag._tok(q)))
for h in mine:
    print(h["id"], h["score"], h["overlap"])


query tokens: ['attachments', 'file', 'overnight', 'store']
HB-ATTACH-04 4 ['attachments', 'file', 'overnight', 'store']
